### Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

#### Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
 - Answer relevance
 - Answer accuracy
 - Retrieval quality
 
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

### ChatBot Evaluation


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
BAZARR_API_KEY = os.getenv("BAZARRLINK")

if not LANGSMITH_API_KEY:
    raise RuntimeError("Missing LANGSMITH_API_KEY. Add it to your .env file.")

if not BAZARR_API_KEY:
    raise RuntimeError("Missing BAZARRLINK. Add your API key to your .env file.")

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_TRACING"] = "true"


In [2]:
# Create or reuse the evaluation dataset.
from langsmith import Client

ls_client = Client()
dataset_name = "Chatbots Evaluation"

try:
    dataset = ls_client.read_dataset(dataset_name=dataset_name)
    print(f"Using existing dataset: {dataset_name}")
except Exception as exc:
    # Only create the dataset if it does not already exist.
    # If this fails due to authentication/network issues, the exception will surface.
    from langsmith.utils import LangSmithNotFoundError
    if not isinstance(exc, LangSmithNotFoundError):
        raise
    dataset = ls_client.create_dataset(dataset_name=dataset_name)
    ls_client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {"inputs": {"question": "What is LangChain?"},
             "outputs": {"answer": "A framework for building LLM applications"}},
            {"inputs": {"question": "What is LangSmith?"},
             "outputs": {"answer": "A platform for observing and evaluating LLM applications"}},
            {"inputs": {"question": "What is OpenAI?"},
             "outputs": {"answer": "A company that develops AI models and products"}},
            {"inputs": {"question": "What is Google?"},
             "outputs": {"answer": "A technology company known for search and other products"}},
            {"inputs": {"question": "What is Mistral?"},
             "outputs": {"answer": "An AI company that develops language models"}},
        ],
    )

print(f"Dataset ID: {dataset.id}")


Using existing dataset: Chatbots Evaluation
Dataset ID: c25008cb-9523-4cfc-a12c-56c4c87a6e6c


### Define Metrics (LLM as a Judge)

In [3]:
import random
import time
from openai import OpenAI, APIError
from langsmith import wrappers

client = OpenAI(
    api_key=BAZARR_API_KEY,
    base_url="https://api.bazaarlink.ai/v1",
)
# Keep LangSmith tracing for supported OpenAI calls.
wrapped_client = wrappers.wrap_openai(client)

last_request_at = 0.0

def generate_content_with_retry(
    *,
    model: str,
    contents: str,
    system_instruction: str,
    temperature: float = 0,
    max_retries: int = 5,
):
    """Call an OpenAI-compatible chat-completions endpoint with retries."""
    global last_request_at
    retryable_status_codes = {429, 500, 502, 503, 504}
    min_interval = 12.0

    for attempt in range(max_retries):
        elapsed = time.monotonic() - last_request_at
        if elapsed < min_interval:
            time.sleep(min_interval - elapsed)

        try:
            response = wrapped_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": contents},
                ],
                temperature=temperature,
            )
            last_request_at = time.monotonic()
            return response.choices[0].message.content or ""
        except APIError as error:
            last_request_at = time.monotonic()
            status_code = getattr(error, "status_code", None)
            if status_code not in retryable_status_codes or attempt == max_retries - 1:
                raise

            delay = min(2 ** attempt, 30) + random.uniform(0, 1)
            if status_code == 429:
                delay = max(delay, 30)
            print(
                f"API error (HTTP {status_code}). "
                f"Retry {attempt + 1}/{max_retries - 1} in {delay:.1f}s."
            )
            time.sleep(delay)

    raise RuntimeError("Request failed after all retries.")


In [4]:
import re

EVALUATOR_MODEL = "qwen/qwen3.7-flash"

eval_instructions = """You are a strict but fair evaluator of chatbot answers.
Compare the predicted answer with the reference answer. Judge factual correctness
and relevance to the question. The wording need not exactly match the reference.
Return exactly one word: CORRECT or INCORRECT. Do not add explanation or punctuation.
"""

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    question = inputs.get("question", "")
    reference = reference_outputs.get("answer", "")
    prediction = outputs.get("response", "")

    if not prediction:
        return False

    user_content = (
        f"Question:\n{question}\n\n"
        f"Reference answer:\n{reference}\n\n"
        f"Predicted answer:\n{prediction}\n\n"
        "Is the predicted answer factually correct and relevant?"
    )

    response = generate_content_with_retry(
        model=EVALUATOR_MODEL,
        contents=user_content,
        system_instruction=eval_instructions,
        temperature=0,
    )
    verdict = re.sub(r"[^A-Z]", "", response.strip().upper())

    if verdict == "CORRECT":
        return True
    if verdict == "INCORRECT":
        return False
    raise ValueError(f"Unexpected evaluator response: {response!r}")


In [5]:
# Simple length-based heuristic: output must be shorter than 2x reference length.
def concision(outputs: dict, reference_outputs: dict) -> bool:
    prediction = str(outputs.get("response", "")).strip()
    reference = str(reference_outputs.get("answer", "")).strip()
    if not prediction or not reference:
        return False
    return len(prediction) < 2 * len(reference)


### Run evaluation

In [6]:
DEFAULT_INSTRUCTIONS = (
    "Answer the user's question accurately in one short, concise sentence."
)

def my_app(
    question: str,
    model: str = "qwen/qwen3.7-flash",
    instructions: str = DEFAULT_INSTRUCTIONS,
) -> str:
    return generate_content_with_retry(
        model=model,
        contents=question,
        system_instruction=instructions,
        temperature=0,
    )


In [7]:
# LangSmith target: inputs are dictionaries containing the dataset's "question" key.
def ls_target(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"])}


In [20]:
# Evaluate Qwen.
qwen_experiment_results = ls_client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision], # pyright: ignore[reportArgumentType]
    experiment_prefix="qwen-qwen3.7-flash-evaluation",
    max_concurrency=1,
    error_handling="ignore",
)


View the evaluation results for experiment: 'qwen-qwen3.7-flash-evaluation-2518396a' at:
https://smith.langchain.com/o/3eb47f54-a078-431a-862e-ae8747a3f295/datasets/c25008cb-9523-4cfc-a12c-56c4c87a6e6c/compare?selectedSessions=2afd4624-3c22-4986-8e03-5592e2b1aeb2




5it [01:17, 15.46s/it]


In [8]:
# Target for the second model; keep a separate function so evaluations do not overwrite each other.
def deepseek_target(inputs: dict) -> dict:
    return {
        "response": my_app(
            inputs["question"],
            model="deepseek/deepseek-v4-flash-0731free",
            instructions=(
                "Answer the user's question accurately in one short, concise sentence."
            ),
        )
    }


In [22]:
# Evaluate DeepSeek.
deepseek_experiment_results = ls_client.evaluate(
    deepseek_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="deepseek-v4-flash-0731free-evaluation",
    max_concurrency=1,
    error_handling="ignore",
) # pyright: ignore[reportCallIssue]


View the evaluation results for experiment: 'deepseek-v4-flash-0731free-evaluation-f8f589af' at:
https://smith.langchain.com/o/3eb47f54-a078-431a-862e-ae8747a3f295/datasets/c25008cb-9523-4cfc-a12c-56c4c87a6e6c/compare?selectedSessions=b89cff57-cd7b-4c8e-a74b-194fe282aca7




5it [01:42, 20.48s/it]


In [9]:
# Optional: inspect the experiment result objects in the notebook.
print("Qwen evaluation:", qwen_experiment_results)
print("DeepSeek evaluation:", deepseek_experiment_results)


NameError: name 'qwen_experiment_results' is not defined

### Evaluation of RAG

In [10]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize with a model name (e.g., all-MiniLM-L6-v2)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},  # or "cuda" for GPU
    encode_kwargs={"normalize_embeddings": True} # Recommended for cosine similarity
)

# List of URLS to load and embed
urls = [
    "https://lilianweng.github.io/posts/2023-01-10-inference-optimization",
    "https://lilianweng.github.io/posts/2026-06-24-scaling-laws",
    "https://lilianweng.github.io/posts/2026-07-04-harness",
    "https://lilianweng.github.io/posts/2023-06-23-agent",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering",
    "https://lilianweng.github.io/posts/2024-07-07-hallucination",
]

#Load documents from the web
docs=[WebBaseLoader(url).load() for url in urls]

docs_list = [doc for sublist in docs for doc in sublist]  # Flatten the list of lists

# Save each document
os.makedirs("saved_pages", exist_ok=True)

for i, doc in enumerate(docs_list):
    # Use the URL path as filename for clarity
    slug = doc.metadata["source"].rstrip("/").split("/")[-1]
    filename = f"saved_pages/{slug}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(doc.page_content)
    print(f"Saved: {filename}")
    
print(f"Total documents: {len(docs_list)}")

print(f"Total characters: {sum(len(doc.page_content) for doc in docs_list)}")

print(f"Total tokens: {sum(len(doc.page_content.split()) for doc in docs_list)}")

#Initialize a text splinter with specified chunks size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=0
)



#split th documents into chunks
docs_chunks = text_splitter.split_documents(docs_list)
print(f"Total chunks: {len(docs_chunks)}")
print(f"Total characters in chunks: {sum(len(chunk.page_content) for chunk in docs_chunks)}")
print(f"Total tokens in chunks: {sum(len(chunk.page_content.split()) for chunk in docs_chunks)}")

#Add the documents chunks to the vector store using the embeddings
vector_store = InMemoryVectorStore.from_documents(docs_chunks, embeddings)


#with langchain er can easily turns any vector store into a retrieval components 
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

C:\Users\HP\AppData\Local\Temp\ipykernel_25548\2347198760.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
d:\ProjectsGit\ai-ml\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2611.08it/s]


Saved: saved_pages/2023-01-10-inference-optimization.txt
Saved: saved_pages/2026-06-24-scaling-laws.txt
Saved: saved_pages/2026-07-04-harness.txt
Saved: saved_pages/2023-06-23-agent.txt
Saved: saved_pages/2023-03-15-prompt-engineering.txt
Saved: saved_pages/2024-07-07-hallucination.txt
Total documents: 6
Total characters: 242497
Total tokens: 35684
Total chunks: 341
Total characters in chunks: 241678
Total tokens in chunks: 35684


In [11]:
retriever.invoke("what is agent?")

[Document(id='e9e6be15-4d9c-4251-93fc-466f8b5f6a6c', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, t

In [19]:
##llm chain that uses the retriever to get relevant documents and then answer the question based on those documents

from langchain.chat_models import init_chat_model
llm = init_chat_model(
    model="qwen/qwen3.7-flash",
    model_provider="openai",
    model_kwargs={"temperature": 0},
    api_key=BAZARR_API_KEY,
    base_url="https://api.bazaarlink.ai/v1",
    
)
llm

d:\ProjectsGit\ai-ml\RAG\.venv\Lib\site-packages\langchain\chat_models\base.py:516: UserWarning: Parameters {'temperature'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2', 'langchain-openai': '1.6.6'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000017B505C1E50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000017B505C2210>, root_client=<openai.OpenAI object at 0x0000017B505C1BD0>, root_async_client=<openai.AsyncOpenAI object at 0x0000017B505C1F90>, model_name='qwen/qwen3.7-flash', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.bazaarlink.ai/v1', stream_chunk_timeout=120.0)

In [20]:
from langsmith import traceable

## Add decorator to make the function traceable in LangSmith

@traceable
def rag_bot(question: str) -> dict:
    # Retrieve relevant documents
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Create a prompt that includes the context and the question
    instructions = ("you are a helpful assistant who is good at analyzing soiurces and providing accurate answers. "
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer the question based on the context provided."
    )
    
    # Generate an answer using the LLM
    answer = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ])
    
    return {"response": answer.content,"documents":docs}

In [21]:
rag_bot("what is agent?")

{'response': 'Based on the provided context, an **agent** is described through several key characteristics and architectural components:\n\n1.  **Planning for Believability:** Planning is used to "optimize believability at the moment vs in time."\n2.  **Social and Environmental Awareness:** Agents take into consideration relationships between agents and observations of one agent by another when planning and reacting.\n3.  **Environment Structure:** The environment information is present in a tree structure.\n4.  **Generative Architecture:** It involves a generative agent architecture (referencing Park et al. 2023).\n5.  **Tool Use Loop:** In a simplified loop (such as the Codex agent), the agent calls tools, and the responses from those tools affect the model\'s next generation.\n6.  **LLM-Powered:** The context references "LLM-powered Autonomous Agents," indicating that Large Language Models are central to their operation.',
 'documents': [Document(id='e9e6be15-4d9c-4251-93fc-466f8b5f

### Dataset

In [ ]:
from langchain imposrt